# zagg on CryoCloud: end-to-end example

This notebook demonstrates running the [`zagg`](https://github.com/englacial/xagg) library on CryoCloud, including:

1. **Installing** `zagg` and its dependencies into the CryoCloud user environment
2. **Reading ISMIP6 data** from a virtualizarr / icechunk repository (single variable, DataTree, and inter-model variance with a time slider)
3. **Local processing** of ICESat-2 ATL06 data with a custom aggregation
4. **Lambda processing** of the full ATL06 dataset against an AWS Lambda function in account `429435741471`
5. **Reprojecting and plotting** the result on an Antarctic Polar Stereographic grid (EPSG:3031) using `imshow`

AWS authentication is handled automatically by the CryoCloud IRSA role (`nasa-cryo-prod`) -- no AWS credentials need to be configured manually. NASA Earthdata authentication still uses `earthaccess` (your `~/.netrc` or `earthaccess.login()`).

## 0. Install zagg and configure the Lambda target

On CryoCloud, the bare function name `process-morton-cell` would resolve in CryoCloud's own AWS account (574251165169) and fail with `AccessDeniedException`. We point boto3 at the full cross-account ARN via the `ZAGG_LAMBDA_FUNCTION_NAME` env var that `zagg.runner` reads.

Lambda output is written to the public `xagg` bucket in the zagg owner's account -- the function's execution role already has write permission there, and the result stays colocated with the code that produced it.

In [ ]:
%pip install --quiet 'zagg[analysis] @ git+https://github.com/englacial/xagg.git'
%pip install --quiet icechunk 'cubed-xarray>=0.0.9' hvplot

In [ ]:
import logging
import os

# Show INFO-level zagg messages (per-cell progress, summary lines).
# Without this, only WARNING+ surfaces and runs look silent.
logging.basicConfig(level=logging.INFO, format='%(message)s')

os.environ['ZAGG_LAMBDA_FUNCTION_NAME'] = (
    'arn:aws:lambda:us-west-2:429435741471:function:process-morton-cell'
)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')

OUTPUT_BUCKET = 'xagg'
OUTPUT_KEY = 'atl06/morton_aggregation_cryocloud.zarr'
OUTPUT_URL = f's3://{OUTPUT_BUCKET}/{OUTPUT_KEY}'
print(f'Lambda output -> {OUTPUT_URL}')

# Confirm IRSA credentials are present (sanity check)
import boto3
print(boto3.client('sts').get_caller_identity()['Arn'])

## 1. Reading ISMIP6 data from the virtualizarr / icechunk store

The ISMIP6 Antarctic ensemble is published as an [icechunk](https://icechunk.io) repository at `s3://us-west-2.opendata.source.coop/englacial/ismip6/icechunk-ais`. The icechunk session contains *virtual* references to the original NetCDF chunks plus rechunked overlays.

We open it once anonymously and then show three access patterns:

- **A single variable** from one model/experiment (the most common pattern)
- **The full repository as an `xarray.DataTree`** (when you want to traverse the model/experiment hierarchy programmatically)
- **An inter-model variance** of `lithk` for `exp05`, with a time slider

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='Numcodecs codecs are not in the Zarr version 3')

import icechunk
import xarray as xr
import zarr

SOURCE_BUCKET = 's3://us-west-2.opendata.source.coop/englacial/ismip6'

storage = icechunk.s3_storage(
    bucket='us-west-2.opendata.source.coop',
    prefix='englacial/ismip6/icechunk-ais',
    region='us-west-2',
    anonymous=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        SOURCE_BUCKET + '/',
        store=icechunk.s3_store(region='us-west-2', anonymous=True),
    )
)
credentials = icechunk.containers_credentials({SOURCE_BUCKET + '/': None})

repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)
session = repo.readonly_session(branch='main')

root = zarr.open(session.store, mode='r')
print(root.tree(level=1))

### 1a. Open a single variable from one model/experiment

Each model/experiment is a zarr group under `combined/<model>/<exp>`. We open one of them and plot the ice thickness `lithk` field. The `combined` store unions state and flux variables onto a single time axis, so the very first timestep can be all-NaN for state variables -- we use `time=1` to be safe.

In [ ]:
import matplotlib.pyplot as plt

ds = xr.open_zarr(
    session.store,
    group='combined/AWI_PISM1/exp05',
    consolidated=False,
)
print(f'Variables: {sorted(ds.data_vars)}')
print(f'Time range: {str(ds.time.values[0])[:10]} -> {str(ds.time.values[-1])[:10]} ({len(ds.time)} steps)')

fig, ax = plt.subplots(figsize=(7, 7))
ds['lithk'].isel(time=1).plot(ax=ax, cmap='viridis')
ax.set_aspect('equal')
ax.set_title(f"AWI_PISM1 / exp05 / lithk @ {str(ds.time.values[1])[:10]}")
plt.tight_layout()
plt.show()

### 1b. Open the entire repository as an xarray DataTree

`xr.open_datatree` traverses every group below the root, giving you a nested tree you can iterate or index into with paths. We pass `chunked_array_type="cubed"` so dask is not required (`cubed` is the default chunked backend for icechunk virtual stores).

In [ ]:
ds_tree = xr.open_datatree(
    session.store,
    engine='zarr',
    create_default_indexes=False,
    decode_times=False,
    chunked_array_type='cubed',
)

# Show the top-level groups
print(ds_tree)

# Index into it just like a dict-of-datasets
lithk_awi = ds_tree['combined/AWI_PISM1/exp05'].ds['lithk']
print(f'\nlithk shape: {lithk_awi.shape}, dtype: {lithk_awi.dtype}')

### 1c. Inter-model variance of `lithk` for `exp05` with a time slider

Now we use the DataTree to walk every model under `combined/`, pull the `lithk` field for `exp05`, stack them along a new `model` dimension, and compute variance across models.

ISMIP6 specifies a common 8 km Antarctic grid (761 x 761), so most submissions share spatial coordinates. We restrict to models that match the reference grid and the shortest common time axis; mismatches are skipped with a printed warning. The result is a `(time, y, x)` array that we plot with `hvplot`, which renders a year-by-year time slider.

In [ ]:
# Walk every model in combined/, pull lithk for exp05
model_arrays = []
model_names = []
for model in sorted(ds_tree['combined'].children):
    try:
        node = xr.open_zarr(
            session.store, group=f'combined/{model}/exp05', consolidated=False,
        )
    except (KeyError, FileNotFoundError, ValueError) as e:
        print(f'skip {model}: {e}')
        continue
    if 'lithk' not in node.data_vars:
        print(f'skip {model}: no lithk')
        continue
    model_arrays.append(node['lithk'])
    model_names.append(model)

# Restrict to the reference grid (first model) and the shortest common time axis
ref = model_arrays[0]
ref_shape = (ref.sizes['y'], ref.sizes['x'])
min_time = min(a.sizes['time'] for a in model_arrays)

aligned = []
kept_names = []
for name, arr in zip(model_names, model_arrays):
    if (arr.sizes['y'], arr.sizes['x']) != ref_shape:
        print(f'skip {name}: grid {arr.sizes["y"]}x{arr.sizes["x"]} != {ref_shape}')
        continue
    aligned.append(
        arr.isel(time=slice(0, min_time))
           .assign_coords(x=ref.x, y=ref.y, time=ref.time.isel(time=slice(0, min_time)))
    )
    kept_names.append(name)

print(f'\nUsing {len(aligned)} models, time={min_time} steps, grid={ref_shape}')
print(f'Models: {kept_names}')

model_dim = xr.DataArray(kept_names, dims='model')
stacked = xr.concat(aligned, dim=model_dim, compat='override', coords='minimal')
lithk_var = stacked.var(dim='model').compute()
lithk_var.name = 'lithk_variance'
lithk_var.attrs['units'] = 'm^2'
lithk_var

In [ ]:
import hvplot.xarray  # noqa: F401  (registers the .hvplot accessor)

lithk_var.hvplot.image(
    x='x', y='y',
    groupby='time',
    cmap='viridis',
    clim=(0, float(lithk_var.quantile(0.99))),
    aspect='equal',
    frame_width=500,
    widget_type='scrubber',
    widget_location='bottom',
    title='Inter-model variance of lithk (exp05)',
)

## 2. Local processing with a custom aggregation

Before scaling out to Lambda, validate the pipeline locally on a tiny slice. We start from the bundled `atl06` config and override the aggregation block to compute a leaner set of statistics (median, min/max, variance) instead of the default weighted-mean + quantile bundle.

We use the HTTPS driver here so this also works from non-AWS environments.

First we generate the granule catalog used by both the local and Lambda runs.

In [ ]:
# Generate catalog: ICESat-2 ATL06, cycle 22, parent morton order 6
# Output: catalog_ATL06_cycle22_order6.json in the cwd
!python -m zagg.catalog --cycle 22 --parent-order 6

In [ ]:
import yaml
from zagg import agg
from zagg.config import load_config_from_dict, get_agg_fields

custom_yaml = """
data_source:
  reader: h5coro
  groups: [gt1l, gt1r, gt2l, gt2r, gt3l, gt3r]
  coordinates:
    latitude: "/{group}/land_ice_segments/latitude"
    longitude: "/{group}/land_ice_segments/longitude"
  variables:
    h_li: "/{group}/land_ice_segments/h_li"
    s_li: "/{group}/land_ice_segments/h_li_sigma"
  quality_filter:
    dataset: "/{group}/land_ice_segments/atl06_quality_summary"
    value: 0

aggregation:
  coordinates:
    cell_ids: {dtype: uint64, fill_value: 0}
    morton:   {dtype: int64,  fill_value: 0}
  variables:
    count:      {function: len,    source: h_li, dtype: int32, fill_value: 0}
    h_median:   {function: median, source: h_li, dtype: float32}
    h_min:      {function: min,    source: h_li, dtype: float32}
    h_max:      {function: max,    source: h_li, dtype: float32}
    h_variance: {function: var,    source: h_li, dtype: float32}

output:
  grid:
    type: healpix
    indexing_scheme: nested
    child_order: 12
"""

cfg_custom = load_config_from_dict(yaml.safe_load(custom_yaml))

for name, meta in get_agg_fields(cfg_custom).items():
    func = meta.get('function', meta.get('expression', 'N/A'))
    print(f'{name:12s} -> {func}')

In [ ]:
# Run on 2 cells using HTTPS to confirm the pipeline works end-to-end
results_local = agg(
    cfg_custom,
    catalog='catalog_ATL06_cycle22_order6.json',
    store='./local_test.zarr',
    backend='local',
    driver='https',
    max_cells=2,
    max_workers=2,
    overwrite=True,
)

print(f"Cells processed:    {results_local['cells_with_data']}")
print(f"Total observations: {results_local['total_obs']:,}")
print(f"Wall time:          {results_local['wall_time_s']:.1f}s")

## 3. Full-dataset Lambda run on CryoCloud

With the pipeline validated, we now use the default `atl06` config (which produces the richer set of variables expected by the existing rasterized output) and dispatch every parent cell in the catalog to the Lambda function. The function runs in *your* AWS account (`429435741471`), so all compute and S3 PUT charges bill there -- not to CryoCloud.

Output is written to `s3://xagg/...`. The Lambda's execution role (`zagg-lambda-execution`) is already scoped to that bucket, so writes work without any cross-account bucket-policy work.

The IRSA role `nasa-cryo-prod` was granted `lambda:InvokeFunction` on the function ARN via the merged 2i2c PR; the Lambda's resource-based policy was extended to trust that role via `aws lambda add-permission`.

In [ ]:
from zagg import default_config

cfg_full = default_config('atl06')

results_lambda = agg(
    cfg_full,
    catalog='catalog_ATL06_cycle22_order6.json',
    store=OUTPUT_URL,           # s3://xagg/atl06/morton_aggregation_cryocloud.zarr
    backend='lambda',
    driver='s3',                # Lambda runs in us-west-2; direct S3 read is cheaper
    max_workers=1700,           # Reserved-concurrency cap on the function
    overwrite=True,
)

print(f"Cells with data:    {results_lambda['cells_with_data']:,}")
print(f"Total observations: {results_lambda['total_obs']:,}")
print(f"Wall time:          {results_lambda['wall_time_s']:.1f}s")
print(f"Lambda compute:     {results_lambda['lambda_time_s']:.0f}s")
print(f"Estimated cost:     ${results_lambda['estimated_cost_usd']:.2f}")

## 4. Reproject to EPSG:3031 and plot with imshow

The result is a HEALPix-indexed zarr at order 12 (~3.4 km cells). To make it map-friendly we:

1. Convert each filled HEALPix cell center to lon/lat with `mortie._healpix.pix2ang`
2. Project to Antarctic Polar Stereographic (EPSG:3031) using `pyproj`
3. Bin onto an 8 km regular grid (`np.bincount` for sums; weighted mean for mean elevation; error propagation for sigma)
4. Display each variable with `imshow` under a `cartopy.crs.SouthPolarStereo` axis

In [ ]:
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mortie._healpix import pix2ang
from pyproj import Transformer
from obstore.store import S3Store
from zarr.storage import ObjectStore

CHILD_ORDER = 12
NSIDE = 2 ** CHILD_ORDER

# xagg is public, so an unsigned read works from anywhere
s3 = S3Store(
    OUTPUT_BUCKET,
    prefix=OUTPUT_KEY,
    region='us-west-2',
    skip_signature=True,
)
store = ObjectStore(store=s3, read_only=True)
ds = xr.open_dataset(
    store, engine='zarr', consolidated=True, zarr_format=3,
    group=str(CHILD_ORDER),
)

has_data = ds['count'].values > 0
cell_ids = ds['cell_ids'].values[has_data]
count    = ds['count'].values[has_data].astype(np.float64)
h_mean   = ds['h_mean'].values[has_data].astype(np.float64)
h_sigma  = ds['h_sigma'].values[has_data].astype(np.float64)
h_min    = ds['h_min'].values[has_data].astype(np.float64)
h_max    = ds['h_max'].values[has_data].astype(np.float64)

print(f'Filled HEALPix cells: {has_data.sum():,} / {has_data.size:,}')
print(f'Total observations:   {count.sum():.0f}')

In [ ]:
# Reproject HEALPix cell centers to EPSG:3031
lon, lat = pix2ang(NSIDE, cell_ids.astype(np.int64))
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:3031', always_xy=True)
x, y = transformer.transform(lon, lat)

# Define an 8 km regular grid covering ~south of 59 S
GRID_SPACING = 8000
EXTENT       = 3_400_000
x_min, x_max = -EXTENT, EXTENT
y_min, y_max = -EXTENT, EXTENT
nx = int((x_max - x_min) / GRID_SPACING)
ny = int((y_max - y_min) / GRID_SPACING)
n_out = nx * ny

ix = np.floor((x - x_min) / GRID_SPACING).astype(np.int64)
iy = np.floor((y - y_min) / GRID_SPACING).astype(np.int64)
inside = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
ix, iy = ix[inside], iy[inside]
flat = (iy * nx + ix).astype(np.int64)

count_in  = count[inside]
h_mean_in = h_mean[inside]
h_sigma_in = h_sigma[inside]
h_min_in  = h_min[inside]
h_max_in  = h_max[inside]

# Aggregate onto the regular grid
count_out = np.bincount(flat, weights=count_in, minlength=n_out)
filled = count_out > 0

def weighted_mean(values):
    wsum = np.bincount(flat, weights=values * count_in, minlength=n_out)
    out = np.full(n_out, np.nan)
    out[filled] = wsum[filled] / count_out[filled]
    return out

h_mean_out = weighted_mean(h_mean_in)

h_sigma_out = np.full(n_out, np.nan)
sigma_sq_wsum = np.bincount(flat, weights=count_in**2 * h_sigma_in**2, minlength=n_out)
h_sigma_out[filled] = np.sqrt(sigma_sq_wsum[filled]) / count_out[filled]

h_min_out = np.full(n_out, np.inf)
np.minimum.at(h_min_out, flat, h_min_in)
h_min_out[~filled] = np.nan

h_max_out = np.full(n_out, -np.inf)
np.maximum.at(h_max_out, flat, h_max_in)
h_max_out[~filled] = np.nan

ds_out = xr.Dataset(
    {
        'count':   (['y', 'x'], count_out.reshape(ny, nx).astype(np.int64)),
        'h_mean':  (['y', 'x'], h_mean_out.reshape(ny, nx).astype(np.float32)),
        'h_sigma': (['y', 'x'], h_sigma_out.reshape(ny, nx).astype(np.float32)),
        'h_min':   (['y', 'x'], h_min_out.reshape(ny, nx).astype(np.float32)),
        'h_max':   (['y', 'x'], h_max_out.reshape(ny, nx).astype(np.float32)),
    },
    coords={
        'x': np.arange(x_min + GRID_SPACING / 2, x_max, GRID_SPACING),
        'y': np.arange(y_min + GRID_SPACING / 2, y_max, GRID_SPACING),
    },
    attrs={'crs': 'EPSG:3031', 'grid_spacing_m': GRID_SPACING},
)

In [ ]:
# Plot the four panels with imshow under SouthPolarStereo
proj = ccrs.SouthPolarStereo()
img_extent = [x_min, x_max, y_min, y_max]
mask = ds_out['count'].values > 0

fig, axes = plt.subplots(2, 2, figsize=(16, 14), subplot_kw={'projection': proj})
for ax in axes.flat:
    ax.coastlines(resolution='50m', linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
    ax.gridlines(draw_labels=False, alpha=0.3)
    ax.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())

panels = [
    (axes[0, 0], 'h_mean', 'Mean elevation (m)',     dict(cmap='terrain', vmin=0, vmax=4000)),
    (axes[0, 1], 'count',  'Observation count',       dict(cmap='viridis', norm=LogNorm(vmin=1))),
    (axes[1, 0], 'h_sigma','Combined uncertainty (m)',dict(cmap='plasma', vmax=1.0)),
    (axes[1, 1], None,     'Elevation range (m)',     dict(cmap='hot', vmax=100)),
]

for ax, var, title, kwargs in panels:
    if var is None:
        data = (ds_out['h_max'].values - ds_out['h_min'].values).astype(np.float64)
    else:
        data = ds_out[var].values.astype(np.float64)
    data[~mask] = np.nan
    im = ax.imshow(data, origin='lower', extent=img_extent, transform=proj,
                   interpolation='nearest', **kwargs)
    ax.set_title(title, fontsize=13, weight='bold')
    plt.colorbar(im, ax=ax, shrink=0.7)

plt.suptitle('ATL06 cycle 22 - 8 km Antarctic Polar Stereographic',
             fontsize=15, weight='bold', y=1.02)
plt.tight_layout()
plt.show()